# Brazo D con el adapter fine-tuneado (Colab T4)

Corre **solo el brazo D** (extractor fine-tuneado + `resolvedor_temporal`) sobre las
45 frases de Benchmark2 usando el adapter LoRA ya entrenado (`checkpoint-300`).
No reentrena nada. Duración total: ~20-25 min.

**Requisitos antes de empezar**
1. Runtime **T4 GPU** (`Entorno de ejecución → Cambiar tipo de entorno de ejecución`).
2. `deliverable2/proyecto.zip` del repo, descargado en tu PC.
3. Acceso directo a la carpeta compartida `qwen3-4b-semantic-lora` en **tu** Mi unidad.

Los brazos A, B y C **no** se corren aquí: ya tienen resultados reales y se miden sin adapter.

In [ ]:
# 1) GPU + dependencias (mismas versiones con las que se entrenó el adapter)
!nvidia-smi -L
%pip install -q "transformers==5.17.0" "trl==0.21.0" "peft==0.21.0" "datasets==5.0.1" "accelerate==1.15.0" bitsandbytes
%pip uninstall -y -q torchao
import torch
assert torch.cuda.is_available(), "Sin GPU: cambia el runtime a T4 y vuelve a correr esta celda"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Subir proyecto.zip y extraerlo (normaliza separadores Windows -> POSIX)
import os, sys, shutil, zipfile
os.chdir("/content")
if not os.path.isfile("/content/proyecto/pipeline/__init__.py"):
    shutil.rmtree("/content/proyecto", ignore_errors=True)
    from google.colab import files
    print("Sube proyecto.zip:")
    zip_path = next(iter(files.upload()))
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            nombre = info.filename.replace("\\", "/")
            if not nombre or nombre.endswith("/") or "__pycache__" in nombre:
                continue
            ruta = os.path.join("/content/proyecto", *nombre.split("/"))
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with z.open(info) as src, open(ruta, "wb") as dst:
                shutil.copyfileobj(src, dst)
RAIZ = "/content/proyecto"
os.chdir(RAIZ); sys.path.insert(0, RAIZ)
assert os.path.isfile("benchmark/benchmark2.json"), "zip incompleto"
print("Proyecto listo en", RAIZ)

In [ ]:
# 3) Montar Drive y ubicar el adapter
from google.colab import drive
drive.mount("/content/drive")
ADAPTER = "/content/drive/MyDrive/qwen3-4b-semantic-lora/checkpoint-300"
for f in ("adapter_config.json", "adapter_model.safetensors"):
    assert os.path.isfile(os.path.join(ADAPTER, f)), (
        "Falta %s en %s. ¿Creaste el acceso directo a la carpeta compartida en 'Mi unidad'?" % (f, ADAPTER))
print("Adapter encontrado:", ADAPTER)

In [ ]:
# 4) Correr el brazo D sobre las 45 frases (~2 min de carga + ~10-15 min de inferencia)
!python experimentos/correr_experimento.py --brazo D --adapter "{ADAPTER}"

In [ ]:
# 5) Vista previa de la métrica (criterio idéntico a D1: evaluar.py sin modificar)
#    Referencia: brazo D con el modelo BASE = 84.4%
import json
!python experimentos/evaluar_todo.py
with open("resultados/resultados_D.json", encoding="utf-8") as f:
    filas = json.load(f)
print("frases:", len(filas), "| entorno:", filas[0].get("entorno"))

In [ ]:
# 6) Respaldo en Drive + descarga (envía estos 2 archivos a Matías)
import shutil
from google.colab import files
destino = "/content/drive/MyDrive/resultados_brazo_D"
os.makedirs(destino, exist_ok=True)
for nombre in ("resultados_D.json", "resultados_D_crudo.json"):
    shutil.copy(os.path.join("resultados", nombre), destino)
    files.download(os.path.join("resultados", nombre))
print("Copia en Drive:", destino)